# Page audit on Colab: which parser each PDF page needs

Run this **after the corpus is downloaded** to Drive (`full` or `min500`; cell 3 can download it first).
For every page of every filing it predicts the cheapest reader that can extract the page, and writes one CSV per PDF
in folders that mirror the downloaded sector folders:

```
MyDrive/capstone_rag/annual_reports/retail/EDGAR_ARS_COST_2025.pdf      (downloaded PDF)
MyDrive/capstone_rag/page_audit/retail/EDGAR_ARS_COST_2025.csv          (this notebook)
    pageNumber,parserUsage
    1,ocr
    2,pdfParser
    ...
MyDrive/capstone_rag/page_audit/_summary.csv     one row per filing: pages per parser, seconds
MyDrive/capstone_rag/page_audit/_summary.json    totals per parser and per sector, failed PDFs, run time
```

| parserUsage | Meaning |
|---|---|
| `pdfParser` | plain text: the PDF text layer is exact and enough |
| `pdfParser+colpali` | table, chart or picture whose figures are in the text layer: parse the text, and index the page image with ColPali |
| `ocr` | printed text with an unusable text layer (garbled fonts, glyphs drawn as vector outlines) or a scanned page |
| `qwen3VL` | chart without printed values, or an infographic: needs a vision-language model, Qwen3-VL (chart values are estimates) |
| `paidVL` | table, chart or picture whose text layer is unusable: nothing to check a model against, use the strongest model |
| `none` | blank page, or a photo/logo with no text and no figures: skip it |

A PDF that can't be opened (corrupt, encrypted, missing) gets no CSV and is listed under `failed` in `_summary.json`.

**Runtime: CPU only.** *Runtime → Change runtime type → CPU* (a GPU doesn't help; Colab Pro "High-RAM" gives 8 CPUs).
Measured at about 0.22 s per page per CPU core:

| | min500 (509 filings, ~80k pages) | full (1,004 filings, ~159k pages) |
|---|---|---|
| standard Colab, 2 CPUs | ~3-4 h | ~6-7.5 h |
| Colab Pro High-RAM, 8 CPUs | ~45-60 min | ~1.5-2 h |

Finished filings are skipped on a rerun, so **after a disconnect run cells 2-4 again**; it continues where it stopped.
Start with `SMOKE_TEST = True` (3 filings, a few minutes): it prints a time estimate for the whole run.

**Code:** on your PC run `python -m scripts.common.colab_bundle` and upload `output/colab/capstone_code.zip` to
`MyDrive/capstone_rag/`. Cell 2 unpacks it; without the zip it clones the repo (needs a `GITHUB_TOKEN` secret while
the repo is private).

In [ ]:
# @title 1. Settings
CORPUS = "min500"  # @param ["min500", "full"]
SECTORS = ""  # @param {type:"string"}
# "" = every downloaded sector; or e.g. "banking,retail"
DOWNLOAD_FIRST = False  # @param {type:"boolean"}
# True: download the CORPUS PDFs that are not on Drive yet (resumable), then audit
SMOKE_TEST = True  # @param {type:"boolean"}
# True: audit the first 3 filings into page_audit_smoke/ and print a time estimate for the whole run
WORKERS = 0  # @param {type:"integer"}
# 0 = one process per CPU of this runtime
FORCE = False  # @param {type:"boolean"}
# True: redo filings that already have a CSV (after changing the routing rules)
SEC_USER_AGENT = "capstone-research your.name@example.com"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/capstone_rag"
REPO = "https://github.com/rama159065/Capstone.git"   # used only when there is no code zip
BRANCH = "financial-report-only"
CODE_ZIP = "/content/drive/MyDrive/capstone_rag/capstone_code.zip"   # python -m scripts.common.colab_bundle, then upload

In [ ]:
# @title 2. Drive, code, dependencies
import json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")


def fetch_code(root, repo_url, branch):
    # Unpack the code zip (Drive, else one you upload) into root; without a zip, clone the repo.
    import zipfile
    uploaded = Path("/content") / Path(CODE_ZIP).name
    source = next((p for p in (Path(CODE_ZIP), uploaded) if p.exists()), None)
    if source is None and not root.exists():
        from google.colab import files
        print(f"{CODE_ZIP} not found: upload capstone_code.zip (cancel to clone {repo_url} instead)")
        names = list(files.upload())
        source = Path("/content") / names[0] if names else None
    if source is not None:
        with zipfile.ZipFile(source) as archive:
            info = archive.read("BUNDLE_INFO.json")
            stamp = root / "BUNDLE_INFO.json"
            if not stamp.exists() or stamp.read_bytes() != info:      # a new build: unpack over the old one
                archive.extractall(root)
        print("code:", source, json.loads(info))
    elif not root.exists():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")      # needed while the repo is private
        except Exception:
            token = None
        url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, str(root)], check=True)


fetch_code(Path("/content/Capstone"), REPO, BRANCH)
sys.path.insert(0, "/content/Capstone")
%cd /content/Capstone
!pip -q install "pymupdf>=1.24,<2" "qdrant-client>=1.12,<2"
from scripts.rag import page_audit
assert "qwen3VL" in page_audit.LABELS, "old code: rebuild capstone_code.zip (python -m scripts.common.colab_bundle) and upload it"
print("CPUs:", os.cpu_count(), "| labels:", ", ".join(page_audit.LABELS))

In [ ]:
# @title 3. Corpus on Drive (optionally download what is missing)
from collections import Counter
from scripts.rag import colab_pipeline as cp
from scripts.rag import financial_ingest_qdrant_db as fi
os.environ["SEC_USER_AGENT"] = SEC_USER_AGENT
s = cp.Settings(corpus_preset=CORPUS, drive_root=DRIVE_ROOT, sectors=SECTORS, user_agent=SEC_USER_AGENT,
                max_per_sector=None)          # None = the whole preset when downloading
if DOWNLOAD_FIRST:
    cp.download(s)
try:
    rows = fi.in_preset(fi.read_manifest(s.corpus, SECTORS or None), CORPUS)
except FileNotFoundError:
    raise SystemExit(f"No manifests under {s.corpus}: set DOWNLOAD_FIRST = True (or run colab_rag_pipelines.ipynb cell 4)")
present = [r for r in rows if any(p.exists() for p in fi.pdf_candidates(r, None))]
print(f"{CORPUS}: {len(rows)} filings in the manifests, {len(present)} PDFs on Drive")
print(dict(sorted(Counter(r["sector_slug"] for r in present).items())))
if len(present) < len(rows):
    print(f"{len(rows) - len(present)} PDFs missing: they will be listed as failed (DOWNLOAD_FIRST = True fetches them)")

In [ ]:
# @title 4. Run the page audit (rerun after a disconnect: cells 2-4; finished filings are skipped)
import time
OUT = Path(DRIVE_ROOT) / ("page_audit_smoke" if SMOKE_TEST else "page_audit")
started = time.time()
summary = cp.page_audit(s, workers=WORKERS or None, force=FORCE, limit=3 if SMOKE_TEST else None, out=OUT)
print(f"\ndone in {(time.time() - started) / 60:.1f} min -> {OUT}")
print(json.dumps({k: summary[k] for k in ("filings", "status", "pages", "parser_usage", "parser_share",
                                          "seconds_per_page")}, indent=2))
if SMOKE_TEST and summary["seconds_per_page"]:
    workers = WORKERS or os.cpu_count()
    pages_per_filing = summary["pages"] / max(1, summary["filings"])
    hours = len(present) * pages_per_filing * summary["seconds_per_page"] / workers / 3600
    print(f"\nEstimate for all {len(present)} downloaded {CORPUS} filings with {workers} worker(s): ~{hours:.1f} h "
          f"({pages_per_filing:.0f} pages/filing x {summary['seconds_per_page']} s/page). "
          "Set SMOKE_TEST = False and run cells 3-4.")

In [ ]:
# @title 5. Results: parser usage per sector and per filing
import pandas as pd
table = pd.read_csv(OUT / "_summary.csv", keep_default_na=False)
labels = list(page_audit.LABELS)
by_sector = table[table.status != "failed"].groupby("sector_slug")[["pages", *labels]].sum()
by_sector.loc["TOTAL"] = by_sector.sum()
display(by_sector)
share = (by_sector.loc["TOTAL", labels] / by_sector.loc["TOTAL", "pages"] * 100).round(1)
print("share of pages (%):", share.to_dict())
failed = table[table.status == "failed"]
if len(failed):
    print(f"\n{len(failed)} filing(s) could not be analysed (see _summary.json -> failed):")
    display(failed[["sector_slug", "document_id"]])
# one filing's CSV (keep_default_na=False keeps the label 'none' as text)
sample = next(OUT.glob("*/*.csv"))
print("\n", sample)
display(pd.read_csv(sample, keep_default_na=False).parserUsage.value_counts())

In [ ]:
# @title 6. (Optional) zip the CSVs to download them
import shutil
archive = shutil.make_archive(f"/content/{OUT.name}", "zip", OUT)
print(archive)
from google.colab import files
files.download(archive)